# Code Agent (Notebook)

A generic agentic assistant that can interact with GitHub repos, Jira issues, and more via MCP tools. Pass any query and the agent will break it down into steps, call the necessary tools, and complete the task autonomously.

- An **OpenAI-compliant** inference endpoint secured with API Key
- **MCP Endpoint** secured with API Key — tools for GitHub (PRs, branches, file ops) and Jira (issues, search)

Configure via `.env`: `OPENAI_BASE_URL`, `OPENAI_API_KEY`, `MCP_SERVER_URL`, `MCP_AUTH_TOKEN`, optionally `OPENAI_MODEL`.

## 1. Imports and config

In [16]:
import json
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

MCP_URL = os.getenv("MCP_SERVER_URL")
MCP_BEARER_TOKEN = os.getenv("MCP_AUTH_TOKEN")
OPENAI_BASE = os.getenv("OPENAI_BASE_URL")
OPENAI_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-oss-20b")

print(f"MCP_URL: {MCP_URL}")
print(f"OPENAI_BASE: {OPENAI_BASE or '(not set)'}")
print(f"OPENAI_MODEL: {OPENAI_MODEL}")

MCP_URL: https://agents.ai.nutanix.com/enterpriseai/mcp/demo
OPENAI_BASE: https://agents.ai.nutanix.com/enterpriseai/v1
OPENAI_MODEL: gpt-oss-20b


## 2. MCP client

In [17]:
MCP_SESSION_ID = [None]
_request_id_counter = [0]


def _next_request_id() -> int:
    _request_id_counter[0] += 1
    return _request_id_counter[0]


def _mcp_headers():
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if MCP_BEARER_TOKEN:
        headers["Authorization"] = f"Bearer {MCP_BEARER_TOKEN}"
    if MCP_SESSION_ID[0]:
        headers["Mcp-Session-Id"] = MCP_SESSION_ID[0]
    return headers


def _capture_session_id(resp) -> None:
    """Extract MCP session ID from response headers (case-insensitive) or body."""
    sid = None
    for header_name in ("Mcp-Session-Id", "mcp-session-id", "MCP-Session-ID"):
        sid = resp.headers.get(header_name)
        if sid:
            break

    if not sid and (resp.text or "").strip():
        try:
            data = _parse_mcp_response_body(resp.text)
            if isinstance(data, dict):
                result = data.get("result", data)
                if isinstance(result, dict):
                    sid = result.get("sessionId") or result.get("session_id")
        except Exception:
            pass
    if sid:
        MCP_SESSION_ID[0] = sid


def _parse_mcp_response_body(text: str):
    """Parse MCP response: plain JSON (object or array) or SSE with multiple events."""
    if not text or not text.strip():
        return {}
    text = text.strip()

    # Plain JSON object or array
    if text.startswith("{") or text.startswith("["):
        return json.loads(text)

    # SSE format: may contain multiple "event: message\ndata: {...}" blocks.
    # Each block is a separate JSON-RPC message; return the last one
    # (typically the final result after any progress notifications).
    if "data:" in text:
        parsed_events = []
        for line in text.split("\n"):
            line = line.strip()
            if line.startswith("data:"):
                payload = line[5:].strip()
                if payload:
                    try:
                        parsed_events.append(json.loads(payload))
                    except json.JSONDecodeError:
                        continue
        if parsed_events:
            # Return the last event that has a "result" or "error" key (the final response),
            # falling back to the very last event.
            for evt in reversed(parsed_events):
                if isinstance(evt, dict) and ("result" in evt or "error" in evt):
                    return evt
            return parsed_events[-1]

    return json.loads(text)


def mcp_request(method: str, params: dict | None = None, _retry: bool = True) -> dict:
    payload = {"jsonrpc": "2.0", "id": _next_request_id(), "method": method}
    if params is not None:
        payload["params"] = params
    headers = _mcp_headers()
    resp = requests.post(MCP_URL, json=payload, headers=headers, timeout=60)
    _capture_session_id(resp)

    # On 400, the session may have expired — re-initialize and retry once.
    if resp.status_code == 400 and _retry and method != "initialize":
        print(f"  ⚠️  Got 400 on '{method}', re-initializing session and retrying…")
        MCP_SESSION_ID[0] = None
        mcp_initialize()
        return mcp_request(method, params, _retry=False)

    resp.raise_for_status()
    try:
        data = _parse_mcp_response_body(resp.text or "")
    except json.JSONDecodeError:
        raise RuntimeError(f"MCP response is not JSON. Status: {resp.status_code}. Body: {(resp.text or '')[:300]!r}")
    if isinstance(data, dict) and "error" in data:
        err = data["error"]
        raise RuntimeError(err.get("message", str(err)) if isinstance(err, dict) else str(err))
    if isinstance(data, dict):
        return data.get("result", data)
    return data


def mcp_initialize():
    MCP_SESSION_ID[0] = None
    _request_id_counter[0] = 0
    result = mcp_request("initialize", {
        "protocolVersion": "2024-11-05",
        "capabilities": {},
        "clientInfo": {"name": "code-review-agent", "version": "1.0"},
    }, _retry=False)
    # Send the required initialized notification to complete the handshake
    try:
        mcp_request("notifications/initialized", _retry=False)
    except Exception:
        pass
    print(f"  ✅ MCP session initialized (session_id={MCP_SESSION_ID[0]!r})")
    return result


def mcp_list_tools() -> list:
    result = mcp_request("tools/list")
    all_tools = result.get("tools", [])
    return all_tools


def mcp_call_tool(name: str, arguments: dict) -> str:
    result = mcp_request("tools/call", {"name": name, "arguments": arguments})
    if isinstance(result, dict):
        for part in result.get("content", []):
            if part.get("type") == "text":
                return part.get("text", "")
    return str(result)

## 3. MCP → OpenAI tools and system prompt

In [18]:
def mcp_tools_to_openai(mcp_tools: list) -> list:
    openai_tools = []
    for t in mcp_tools:
        openai_tools.append({
            "type": "function",
            "function": {
                "name": t["name"],
                "description": t.get("description", ""),
                "parameters": t.get("inputSchema", {"type": "object", "properties": {}}),
            },
        })
    return openai_tools


SYSTEM_PROMPT = """You are a helpful AI agent with access to tools for interacting with GitHub repositories, Jira issues, and more.

When the user asks you to accomplish a task, break it down into steps and use the available tools to complete each step. You may need to call multiple tools in sequence — do so until the task is fully complete. After finishing, reply with a concise summary of what you did."""

## 4. Generic agent loop

In [19]:
import re

_KNOWN_TOOL_NAMES: set[str] = set()

def _sanitize_tool_name(raw_name: str) -> str:
    """Strip special-token leaks (e.g. '<|channel|>commentary') that some models append."""
    cleaned = re.split(r"<\|[^|]*\|>", raw_name)[0].strip()
    if _KNOWN_TOOL_NAMES and cleaned not in _KNOWN_TOOL_NAMES:
        for known in _KNOWN_TOOL_NAMES:
            if cleaned.startswith(known) or known.startswith(cleaned):
                return known
    return cleaned


def _agent_loop(client, messages, openai_tools, max_turns=15):
    """Inner loop: let the LLM call tools across multiple turns until done."""
    final_content = ""
    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=messages,
            tools=openai_tools,
            tool_choice="auto",
        )
        choice = response.choices[0]
        msg = choice.message

        print(f"\n--- Turn {turn + 1} (finish_reason={choice.finish_reason}) ---")
        if msg.content:
            print(f"Assistant: {msg.content}")
            final_content = msg.content.strip()

        tool_calls = getattr(msg, "tool_calls", None) or []
        if not tool_calls:
            return final_content or "Done."

        for tc in tool_calls:
            clean_name = _sanitize_tool_name(tc.function.name)
            print(f"  🔧 {clean_name}({tc.function.arguments[:200]})")

        assistant_msg = {
            "role": "assistant",
            "content": msg.content or None,
            "tool_calls": [
                {"id": tc.id, "type": "function", "function": {"name": _sanitize_tool_name(tc.function.name), "arguments": tc.function.arguments}}
                for tc in tool_calls
            ],
        }
        messages.append(assistant_msg)

        for tc in tool_calls:
            name = _sanitize_tool_name(tc.function.name)
            try:
                args = json.loads(tc.function.arguments) if isinstance(tc.function.arguments, str) else tc.function.arguments
            except json.JSONDecodeError:
                args = {}
            try:
                result = mcp_call_tool(name, args)
            except Exception as e:
                result = json.dumps({"error": str(e)})
            print(f"  ✅ {name} → {result[:300]}...")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    return final_content or "Max turns reached."


def interactive_chat(max_turns_per_query: int = 15):
    """
    Interactive chat loop: continuously prompts for user input,
    runs the agent loop for each query, and keeps conversation history
    so follow-up questions work. Type 'exit' or 'quit' to stop.
    """
    if not OPENAI_BASE:
        raise ValueError("Set OPENAI_API_BASE to your OpenAI-compliant inference endpoint URL")

    base_url = OPENAI_BASE.rsplit("/chat/completions", 1)[0] if "/chat/completions" in OPENAI_BASE else OPENAI_BASE.rstrip("/")
    client = OpenAI(base_url=base_url, api_key=OPENAI_KEY)

    mcp_initialize()
    mcp_tool_list = mcp_list_tools()
    _KNOWN_TOOL_NAMES.clear()
    _KNOWN_TOOL_NAMES.update(t["name"] for t in mcp_tool_list)
    openai_tools = mcp_tools_to_openai(mcp_tool_list)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    print("💬 Interactive Agent Chat (type 'exit' to quit)")
    print("=" * 50)

    while True:
        try:
            user_input = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user_input:
            continue
        if user_input.lower() in ("exit", "quit"):
            print("Goodbye!")
            break

        messages.append({"role": "user", "content": user_input})
        result = _agent_loop(client, messages, openai_tools, max_turns=max_turns_per_query)
        messages.append({"role": "assistant", "content": result})

## 5. Run the agent

In [20]:
interactive_chat()

  ✅ MCP session initialized (session_id='SDlBHhBiIQx2QBqPHusTycCs1Cezm3cw7JGy4QfA8gVav+Dvq3dU5fwF/Fy9t2mM0Z7w3hRK/ho47BfHcCzJVSayAjKn5fbbNttH+bicfIDJVcb7qa2M+lW7vPXBzoj/ygITxQRqnBOR7abmlfKOjkQ8DLIInaCK37OI0gkA2/Zwtw6hXKNOFYRih/N6G+Yi/R7OkZaoNHCSVrtuCeOy82QQ9UsHPeOAuI0FtWXLRXVlypY0I6En9bdGodD5G98Zzy2qfnWRpGDBQb0G7+1YLAqu')
💬 Interactive Agent Chat (type 'exit' to quit)
  ✅ mcp-atlassian-ntnx-1__jira_get_issue → {
  "id": "2540733",
  "key": "NAI-4199",
  "summary": "Security vulnerability in Hritik003/test-repo main branch",
  "description": "In the file https://github.com/Hritik003/test-repo/blob/main/weather.py at line number 3, the WEATHER*API*KEY is exposed in code. It should not be exposed as a consta...

--- Turn 3 (finish_reason=tool_calls) ---
  🔧 mcp-github-local__get_file_contents({"owner": "Hritik003", "repo": "test-repo", "path": "weather.py"})
  ✅ mcp-github-local__get_file_contents → {'name': 'weather.py', 'path': 'weather.py', 'sha': '0198dc44e5ef361bd8ca6fbb0fa0b5fe420e2